# Generate CUI similarity scores
## across species

In [5]:
# run in bash
# conda create -n scispacy_env python=3.11 -y
# conda activate scispacy_env

# pip install scispacy==0.5.5 spacy==3.7.5

# pip install ipykernel
# pip install click
# pip install pandas
# python -m pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_lg-0.5.4.tar.gz
# python -m ipykernel install --user --name scispacy_env --display-name "Python (scispacy_env)"

In [6]:
import json
import pandas as pd
import scispacy
import spacy
from tqdm import tqdm
from scispacy.linking import EntityLinker

In [7]:
# nlp = spacy.load("en_core_sci_lg")

# nlp.add_pipe(
#     "scispacy_linker",
#     config={
#         "resolve_abbreviations": True,
#         "linker_name": "umls"
#     }
# )

# linker = nlp.get_pipe("scispacy_linker")

# def load_metadata_json(path, species_label):
#     with open(path, "r", encoding="utf-8") as f:
#         data = json.load(f)

#     rows = []

#     for gse_id, meta in data.items():
#         title = meta.get("Title", "")
#         summary = meta.get("Summary", "")
#         text = f"{title} {summary}".strip()

#         rows.append({
#             "gse_id": gse_id,
#             "species": species_label,
#             "title": title,
#             "summary": summary,
#             "text": text
#         })

#     return pd.DataFrame(rows)


# def extract_cuis(text, score_threshold=0.80):
#     doc = nlp(text)
#     cuis = set()

#     for ent in doc.ents:
#         for cui, score in ent._.kb_ents:
#             if score >= score_threshold:
#                 cuis.add(cui)

#     return cuis

In [21]:
# 2. read data
# human_df = load_metadata_json("metadata/metadata_human.json", "human")
# mouse_df = load_metadata_json("metadata/metadata_mouse.json", "mouse")

# 3. generate CUIs for each GSE
# first time
# tqdm.pandas()

# human_df["cui_set"] = human_df["text"].progress_apply(extract_cuis)
# mouse_df["cui_set"] = mouse_df["text"].progress_apply(extract_cuis)

# human_df.to_pickle("metadata/human_with_cuis.pkl")
# mouse_df.to_pickle("metadata/mouse_with_cuis.pkl")

# future:
human_df = pd.read_pickle("metadata/human_with_cuis.pkl")
mouse_df = pd.read_pickle("metadata/mouse_with_cuis.pkl")

# 4. calculate CUI overlap similarity
def jaccard(a, b):
    if not a or not b:
        return 0
    return len(a & b) / len(a | b)


def containment(a, b):
    if not a or not b:
        return 0
    return len(a & b) / min(len(a), len(b))


def overlap_count(a, b):
    return len(a & b)

# 5. find highest similarity pairs and save
from pathlib import Path
import pandas as pd
from tqdm import tqdm

output_path = Path("top_1000_human_mouse_cui_pairs.csv")

# future: if file already exists, read it
if output_path.exists():
    top_pairs = pd.read_csv(output_path)
    print(f"Loaded existing file: {output_path}")

# first time: generate and save
else:
    rows = []

    for _, h in tqdm(human_df.iterrows(), total=len(human_df)):
        h_cuis = h["cui_set"]

        for _, m in mouse_df.iterrows():
            m_cuis = m["cui_set"]
            shared = h_cuis & m_cuis

            if len(shared) == 0:
                continue

            rows.append({
                "human_gse": h["gse_id"],
                "mouse_gse": m["gse_id"],
                "jaccard": jaccard(h_cuis, m_cuis),
                "containment": containment(h_cuis, m_cuis),
                "overlap_count": len(shared),
                "shared_cuis": sorted(shared),
                "human_title": h["title"],
                "mouse_title": m["title"]
            })

    pairs_df = pd.DataFrame(rows)

    top_pairs = pairs_df.sort_values(
        ["containment", "overlap_count", "jaccard"],
        ascending=False
    )

    top_pairs.to_csv(output_path, index=False)
    print(f"Saved new file: {output_path}")

# top_pairs.head(10)

Loaded existing file: top_1000_human_mouse_cui_pairs.csv


In [9]:
# to save
# top_pairs.head(1000).to_csv("top_1000_human_mouse_cui_pairs.csv", index=False)

## within species

In [10]:
import heapq
from tqdm import tqdm
import pandas as pd

def jaccard(a, b):
    if not a or not b:
        return 0
    return len(a & b) / len(a | b)

def containment(a, b):
    if not a or not b:
        return 0
    return len(a & b) / min(len(a), len(b))

def get_top_within_pairs(df, label, top_n=100):
    heap = []
    counter = 0
    n = len(df)

    for i in tqdm(range(n), desc=f"within {label}"):
        gse_i = df.iloc[i]
        cui_i = gse_i["cui_set"]

        for j in range(i + 1, n):
            gse_j = df.iloc[j]
            cui_j = gse_j["cui_set"]

            shared = cui_i & cui_j

            if len(shared) == 0:
                continue

            jac = jaccard(cui_i, cui_j)
            cont = containment(cui_i, cui_j)
            overlap = len(shared)

            score_key = (cont, overlap, jac)

            row = {
                "comparison": label,
                "gse_1": gse_i["gse_id"],
                "gse_2": gse_j["gse_id"],
                "containment": cont,
                "jaccard": jac,
                "overlap_count": overlap,
                "shared_cuis": sorted(shared),
                "title_1": gse_i["title"],
                "title_2": gse_j["title"],
            }

            item = (score_key, counter, row)
            counter += 1

            if len(heap) < top_n:
                heapq.heappush(heap, item)
            else:
                heapq.heappushpop(heap, item)

    rows = [item[2] for item in heap]

    return (
        pd.DataFrame(rows)
        .sort_values(["containment", "overlap_count", "jaccard"], ascending=False)
        .reset_index(drop=True)
    )

In [11]:
# human
# top_human_within = get_top_within_pairs(
#     human_df,
#     label="within_human"
# )

# top_human_within.to_csv("top_within_human_cui_pairs.csv", index=False)

top_human_within = pd.read_csv("top_100_within_human_cui_pairs.csv")
print(f"Loaded existing file: top_100_within_human_cui_pairs.csv")

Loaded existing file: top_100_within_human_cui_pairs.csv


In [12]:
# mouse
# top_mouse_within = get_top_within_pairs(
#     mouse_df,
#     label="within_mouse"
# )

# top_mouse_within.to_csv("top_within_mouse_cui_pairs.csv", index=False)

top_mouse_within = pd.read_csv("top_100_within_mouse_cui_pairs.csv")
print(f"Loaded existing file: top_100_within_mouse_cui_pairs.csv")

Loaded existing file: top_100_within_mouse_cui_pairs.csv


In [13]:
# to save
# top_human_within.head(100).to_csv("top_1000_within_human_cui_pairs.csv", index=False)
# top_mouse_within.head(100).to_csv("top_1000_within_mouse_cui_pairs.csv", index=False)

## Together

In [14]:
# all_top_pairs = pd.concat(
#     [
#         top_pairs.assign(comparison="human_mouse"),
#         top_human_within,
#         top_mouse_within
#     ],
#     ignore_index=True
# )

# all_top_pairs.to_csv("top_all_cui_similarity_pairs.csv", index=False)

# top_human_mouse = top_pairs.rename(columns={
#     "human_gse": "gse_1",
#     "mouse_gse": "gse_2",
#     "human_title": "title_1",
#     "mouse_title": "title_2"
# })

# top_human_mouse["comparison"] = "human_mouse"

# all_top_pairs = pd.concat(
#     [
#         top_human_mouse,
#         top_human_within,
#         top_mouse_within
#     ],
#     ignore_index=True
# )

# all_top_pairs.to_csv("top_all_cui_similarity_pairs.csv", index=False)

In [15]:
# to save
# all_top_pairs.head(3000).to_csv("top_3000_all_cui_similarity_pairs.csv", index=False)

all_top_pairs = pd.read_csv("top_all_cui_similarity_pairs.csv")
print(f"Loaded existing file: top_all_cui_similarity_pairs.csv")

Loaded existing file: top_all_cui_similarity_pairs.csv
